In [0]:
%sql
CREATE WIDGET TEXT run_month DEFAULT '2026-01';

In [0]:
%sql

CREATE OR REPLACE TABLE cmpa_insights_internal_schema.payer360_dq_results (
run_month STRING,
table_name STRING,
check_category STRING,
check_name STRING,
status STRING,
failed_row_count BIGINT,
total_row_count BIGINT,
failure_threshold STRING,
comments STRING,
run_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%sql

-- ============================================================
-- L0_SCHEMA CHECK: PAYER_360_HCP_DETAIL
-- ============================================================

WITH expected_columns AS (
  SELECT EXPLODE(ARRAY(
    'territory_id','territory_name','payer_id','payer_name',
    'hcp_npi','hcp_name','hcp_specialty',
    'hco_npi','hco_name',
    'patient_count','claims_count','total_hcos',
    'last_treatment_date','rollup_level'
  )) AS column_name
),

actual_columns AS (
  SELECT LOWER(column_name) AS column_name
  FROM information_schema.columns
  WHERE table_schema = 'cmpa_insights_internal_schema'
    AND table_name = 'payer_360_hcp_detail'
),

missing_cols AS (
  SELECT e.column_name
  FROM expected_columns e
  LEFT JOIN actual_columns a
    ON e.column_name = a.column_name
  WHERE a.column_name IS NULL
),

extra_cols AS (
  SELECT a.column_name
  FROM actual_columns a
  LEFT JOIN expected_columns e
    ON a.column_name = e.column_name
  WHERE e.column_name IS NULL
),

summary AS (
  SELECT
    (SELECT COUNT(*) FROM missing_cols) +
    (SELECT COUNT(*) FROM extra_cols) AS failed_count
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'PAYER_360_HCP_DETAIL',
  'L0_SCHEMA',
  'COLUMN_STRUCTURE_CHECK',
  CASE WHEN failed_count = 0 THEN 'PASS' ELSE 'FAIL' END,
  failed_count,
  (SELECT COUNT(*) FROM expected_columns),
  'Exact column match required',
  'Validates expected schema structure',
  current_timestamp()
FROM summary;

In [0]:
%sql

-- ============================================================
-- L0_SCHEMA CHECK: PAYER_360_HCO_DETAIL
-- ============================================================

WITH expected_columns AS (
  SELECT EXPLODE(ARRAY(
    'territory_id','territory_name','payer_id','payer_name',
    'hco_npi','hco_name',
    'patient_count','claims_count',
    'last_treatment_date','rollup_level'
  )) AS column_name
),

actual_columns AS (
  SELECT LOWER(column_name) AS column_name
  FROM information_schema.columns
  WHERE table_schema = 'cmpa_insights_internal_schema'
    AND table_name = 'payer_360_hco_detail'
),

missing_cols AS (
  SELECT e.column_name
  FROM expected_columns e
  LEFT JOIN actual_columns a
    ON e.column_name = a.column_name
  WHERE a.column_name IS NULL
),

extra_cols AS (
  SELECT a.column_name
  FROM actual_columns a
  LEFT JOIN expected_columns e
    ON a.column_name = e.column_name
  WHERE e.column_name IS NULL
),

summary AS (
  SELECT
    (SELECT COUNT(*) FROM missing_cols) +
    (SELECT COUNT(*) FROM extra_cols) AS failed_count
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'PAYER_360_HCO_DETAIL',
  'L0_SCHEMA',
  'COLUMN_STRUCTURE_CHECK',
  CASE WHEN failed_count = 0 THEN 'PASS' ELSE 'FAIL' END,
  failed_count,
  (SELECT COUNT(*) FROM expected_columns),
  'Exact column match required',
  'Validates expected schema structure',
  current_timestamp()
FROM summary;

In [0]:
%sql

-- ============================================================
-- L0_SCHEMA CHECK: payer360_master
-- ============================================================

WITH expected_columns AS (
  SELECT EXPLODE(ARRAY(
    'territory_id','territory_name','payer_id','payer_name',
    'parent_id','parent_name',
    'payer_market_share_pct','payer_rank',
    'total_lives','total_elaprase_patients',
    'medicare_patients','medicaid_patients',
    'commercial_patients','other_patients',
    'new_elaprase_patients_r1m','new_elaprase_patients_r3m',
    'age_lt_5_yrs','age_5_to_10_yrs','age_11_to_18_yrs','age_gt_18_yrs',
    'total_primary_hcps','total_hcps',
    'total_primary_hcos','total_hcos',
    'total_claims','pharmacy_total_claims',
    'approved_fills','rejected_fills','reversed_fills',
    'elaprase_approval_rate','elaprase_rejection_rate','elaprase_reversed_rate',
    'pie_completed','account_director',
    'rollup_level'
  )) AS column_name
),

actual_columns AS (
  SELECT LOWER(column_name) AS column_name
  FROM information_schema.columns
  WHERE table_schema = 'cmpa_insights_internal_schema'
    AND table_name = 'payer360_master'
),

missing_cols AS (
  SELECT e.column_name
  FROM expected_columns e
  LEFT JOIN actual_columns a
    ON e.column_name = a.column_name
  WHERE a.column_name IS NULL
),

extra_cols AS (
  SELECT a.column_name
  FROM actual_columns a
  LEFT JOIN expected_columns e
    ON a.column_name = e.column_name
  WHERE e.column_name IS NULL
),

summary AS (
  SELECT
    (SELECT COUNT(*) FROM missing_cols) +
    (SELECT COUNT(*) FROM extra_cols) AS failed_count
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'payer360_master',
  'L0_SCHEMA',
  'COLUMN_STRUCTURE_CHECK',
  CASE WHEN failed_count = 0 THEN 'PASS' ELSE 'FAIL' END,
  failed_count,
  (SELECT COUNT(*) FROM expected_columns),
  'Exact column match required',
  'Validates expected schema structure',
  current_timestamp()
FROM summary;

In [0]:
%sql

-- payer360_master
INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'payer360_master',
'L1_STRUCTURAL',
'ROW_COUNT_NON_ZERO',
CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END,
CASE WHEN COUNT(*) = 0 THEN COUNT(*) ELSE 0 END,
COUNT(*),
'> 0 rows',
'Table must not be empty',
current_timestamp()
FROM cmpa_insights_internal_schema.payer360_master;

-- HCP
INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'PAYER_360_HCP_DETAIL',
'L1_STRUCTURAL',
'ROW_COUNT_NON_ZERO',
CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END,
CASE WHEN COUNT(*) = 0 THEN COUNT(*) ELSE 0 END,
COUNT(*),
'> 0 rows',
'HCP table must not be empty',
current_timestamp()
FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL;

-- HCO
INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'PAYER_360_HCO_DETAIL',
'L1_STRUCTURAL',
'ROW_COUNT_NON_ZERO',
CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END,
CASE WHEN COUNT(*) = 0 THEN COUNT(*) ELSE 0 END,
COUNT(*),
'> 0 rows',
'HCO table must not be empty',
current_timestamp()
FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL;

In [0]:
%sql

WITH national AS (
  SELECT
    total_elaprase_patients,
    total_lives
  FROM cmpa_insights_internal_schema.payer360_master
  WHERE rollup_level = 'NATIONAL'
),

base AS (
  SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN total_elaprase_patients > total_lives THEN 1 ELSE 0 END) AS failures
  FROM cmpa_insights_internal_schema.payer360_master
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'payer360_master',
  'L2_LOGICAL',
  'PATIENTS_LE_LIVES',
  CASE WHEN b.failures = 0 THEN 'PASS' ELSE 'FAIL' END,
  b.failures,
  b.total_rows,
  'patients <= total_lives',
  CONCAT(
    'National Total Elaprase Patients: ', n.total_elaprase_patients,
    ', National Total Lives: ', n.total_lives,
    ', Rows Checked: ', b.total_rows
  ),
  current_timestamp()
FROM base b
CROSS JOIN national n;

In [0]:
%sql

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'payer360_master',
'L2_LOGICAL',
'MARKET_SHARE_SUM_100',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
COUNT(*),
'~100% per territory',
'Market share must sum to 100 per territory',
current_timestamp()
FROM (
  SELECT territory_id,
         ROUND(SUM(payer_market_share_pct),2) AS total_share
  FROM cmpa_insights_internal_schema.payer360_master
  WHERE rollup_level='TERRITORY_PAYER'
  GROUP BY territory_id
  HAVING ROUND(SUM(payer_market_share_pct),2) NOT BETWEEN 99.9 AND 100.1
);

In [0]:
%sql

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'CROSS_TABLE',
'L3_RECON',
'NATIONAL_PATIENT_MATCH',
CASE WHEN ABS(a.total - b.total) = 0 THEN 'PASS' ELSE 'FAIL' END,
ABS(a.total - b.total),
a.total,
'master = hcp national',
'National patient count must match',
current_timestamp()
FROM
(
 SELECT total_elaprase_patients AS total
 FROM cmpa_insights_internal_schema.payer360_master
 WHERE rollup_level='NATIONAL'
) a
CROSS JOIN
(
 SELECT patient_count AS total
 FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
 WHERE rollup_level='NATIONAL'
) b;

In [0]:
%sql

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'CROSS_TABLE',
'L3_RECON',
'HCP_HCO_PATIENT_MATCH',
CASE WHEN ABS(a.total - b.total) = 0 THEN 'PASS' ELSE 'FAIL' END,
ABS(a.total - b.total),
a.total,
'hcp national = hco national',
'HCP and HCO national patients must match',
current_timestamp()
FROM
(
 SELECT patient_count AS total
 FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
 WHERE rollup_level='NATIONAL'
) a
CROSS JOIN
(
 SELECT patient_count AS total
 FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL
 WHERE rollup_level='NATIONAL'
) b;

In [0]:
%sql

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'PAYER_360_HCP_DETAIL',
'L4_GRAIN',
'DUPLICATE_GRAIN_CHECK',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
COUNT(*),
'1 row per grain',
'Checks duplicate rows at Territory, Payer, HCP, HCO, Rollup',
current_timestamp()
FROM (
  SELECT territory_id,payer_id,hcp_npi,hco_npi,rollup_level,
         COUNT(*) c
  FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
  GROUP BY 1,2,3,4,5
  HAVING COUNT(*) > 1
);

In [0]:
%sql

-- ============================================================
-- L2_DATA_FRESHNESS: NATIONAL MAX DATE CHECK
-- ============================================================

WITH national_max AS (
  SELECT
    MAX(last_treatment_date) AS max_date,
    SUM(patient_count) AS total_patients
  FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL
  WHERE rollup_level = 'NATIONAL'
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'PAYER_360_HCO_DETAIL',
  'L2_DATA_FRESHNESS',
  'NATIONAL_MAX_TREATMENT_DATE',
  CASE 
    WHEN max_date <= DATE('${end_date}') THEN 'PASS'
    ELSE 'FAIL'
  END,
  CASE 
    WHEN max_date > DATE('${end_date}') THEN 1 ELSE 0
  END,
  total_patients,
  'Max national treatment date must align with end_date',
  CONCAT(
    'Max Treatment Date: ', CAST(max_date AS STRING),
    ', Total Patients: ', total_patients
  ),
  current_timestamp()
FROM national_max;  

In [0]:
%sql

-- ============================================================
-- L2_TIME_PERIOD: HCP vs HCO MAX DATE MATCH
-- ============================================================

WITH hco_max AS (
  SELECT MAX(last_treatment_date) AS max_date
  FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL
  WHERE rollup_level='NATIONAL'
),

hcp_max AS (
  SELECT MAX(last_treatment_date) AS max_date
  FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
  WHERE rollup_level='NATIONAL'
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'CROSS_TABLE',
  'L2_TIME_PERIOD',
  'HCP_HCO_MAX_DATE_MATCH',
  CASE WHEN hco_max.max_date = hcp_max.max_date THEN 'PASS' ELSE 'FAIL' END,
  CASE WHEN hco_max.max_date <> hcp_max.max_date THEN 1 ELSE 0 END,
  1,
  'HCP and HCO max treatment dates must match',
  CONCAT(
    'HCO Max: ', CAST(hco_max.max_date AS STRING),
    ', HCP Max: ', CAST(hcp_max.max_date AS STRING)
  ),
  current_timestamp()
FROM hco_max
CROSS JOIN hcp_max;

In [0]:
%sql

-- ============================================================
-- L2_DATA_FRESHNESS: LAST TREATMENT ALIGNMENT
-- ============================================================

WITH last_tx AS (
  SELECT
    MAX(last_treatment_date) AS max_tx_date,
    COUNT(*) AS total_rows
  FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL
  WHERE rollup_level='NATIONAL'
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'PAYER_360_HCO_DETAIL',
  'L2_DATA_FRESHNESS',
  'MAX_TREATMENT_DATE_CHECK',
  CASE 
    WHEN max_tx_date <= DATE('${end_date}') THEN 'PASS'
    ELSE 'FAIL'
  END,
  CASE 
    WHEN max_tx_date > DATE('${end_date}') THEN 1 ELSE 0
  END,
  total_rows,
  'Max treatment date must be <= end_date',
  CONCAT('Max Treatment Date: ', CAST(max_tx_date AS STRING)),
  current_timestamp()
FROM last_tx;

In [0]:
%sql

WITH territory_sum AS (
  SELECT SUM(total_elaprase_patients) AS territory_total
  FROM cmpa_insights_internal_schema.payer360_master
  WHERE rollup_level = 'TERRITORY_ALL_PAYER'
),

national AS (
  SELECT total_elaprase_patients AS national_total
  FROM cmpa_insights_internal_schema.payer360_master
  WHERE rollup_level = 'NATIONAL'
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'payer360_master',
  'L2_LOGICAL',
  'TERRITORY_SUM_GE_NATIONAL',
  CASE WHEN t.territory_total >= n.national_total
       THEN 'PASS' ELSE 'FAIL' END,
  CASE WHEN t.territory_total < n.national_total
       THEN 1 ELSE 0 END,
  n.national_total,
  'Territory sum should be >= National (no loss)',
  CONCAT('Territory Sum: ', t.territory_total,
         ', National: ', n.national_total),
  current_timestamp()
FROM territory_sum t
CROSS JOIN national n;

In [0]:
%sql

WITH master AS (
  SELECT total_elaprase_patients
  FROM cmpa_insights_internal_schema.payer360_master
  WHERE rollup_level = 'NATIONAL'
),

hcp AS (
  SELECT patient_count
  FROM cmpa_insights_internal_schema.PAYER_360_HCP_DETAIL
  WHERE rollup_level = 'NATIONAL'
),

hco AS (
  SELECT patient_count
  FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL
  WHERE rollup_level = 'NATIONAL'
)

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
  '${run_month}',
  'CROSS_TABLE',
  'L3_RECON',
  'NATIONAL_PATIENT_TRIPLE_MATCH',
  CASE 
    WHEN m.total_elaprase_patients = h.patient_count
     AND m.total_elaprase_patients = c.patient_count
    THEN 'PASS' ELSE 'FAIL'
  END,
  ABS(m.total_elaprase_patients - h.patient_count),
  m.total_elaprase_patients,
  'Master = HCP = HCO at national',
  CONCAT('Master: ', m.total_elaprase_patients,
         ', HCP: ', h.patient_count,
         ', HCO: ', c.patient_count),
  current_timestamp()
FROM master m
CROSS JOIN hcp h
CROSS JOIN hco c;

In [0]:
%sql

INSERT INTO cmpa_insights_internal_schema.payer360_dq_results
SELECT
'${run_month}',
'PAYER_360_HCO_DETAIL',
'L4_GRAIN',
'DUPLICATE_GRAIN_CHECK',
CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END,
COUNT(*),
COUNT(*),
'1 row per grain',
'Checks duplicate rows at Territory, Payer, HCO, Rollup',
current_timestamp()
FROM (
  SELECT territory_id,payer_id,hco_npi,rollup_level,
         COUNT(*) c
  FROM cmpa_insights_internal_schema.PAYER_360_HCO_DETAIL
  GROUP BY 1,2,3,4
  HAVING COUNT(*) > 1
);

In [0]:
%sql

SELECT *
FROM cmpa_insights_internal_schema.payer360_dq_results
ORDER BY run_timestamp DESC, check_category;

In [0]:
-- ============================================================
-- ROLLUP VALIDATION: Territory → Payer → National
-- ============================================================

WITH territory_payer AS (
    SELECT
        territory_id,
        SUM(total_elaprase_patients) AS territory_sum_from_payers
    FROM cmpa_insights_internal_schema.payer360_master
    WHERE rollup_level = 'TERRITORY_PAYER'
    GROUP BY territory_id
),

territory_all AS (
    SELECT
        territory_id,
        total_elaprase_patients AS territory_direct_total
    FROM cmpa_insights_internal_schema.payer360_master
    WHERE rollup_level = 'TERRITORY_ALL_PAYER'
),

payer_all AS (
    SELECT
        payer_id,
        total_elaprase_patients AS payer_direct_total
    FROM cmpa_insights_internal_schema.payer360_master
    WHERE rollup_level = 'PAYER_ALL_TERRITORY'
),

payer_from_territory AS (
    SELECT
        payer_id,
        SUM(total_elaprase_patients) AS payer_sum_from_territories
    FROM cmpa_insights_internal_schema.payer360_master
    WHERE rollup_level = 'TERRITORY_PAYER'
    GROUP BY payer_id
),

national AS (
    SELECT
        total_elaprase_patients AS national_total
    FROM cmpa_insights_internal_schema.payer360_master
    WHERE rollup_level = 'NATIONAL'
)

SELECT
    'TERRITORY_CHECK' AS check_type,
    t.territory_id,
    t.territory_sum_from_payers,
    a.territory_direct_total,
    (t.territory_sum_from_payers - a.territory_direct_total) AS diff
FROM territory_payer t
LEFT JOIN territory_all a
    ON t.territory_id = a.territory_id

UNION ALL

SELECT
    'PAYER_CHECK' AS check_type,
    p.payer_id,
    p.payer_sum_from_territories,
    a.payer_direct_total,
    (p.payer_sum_from_territories - a.payer_direct_total) AS diff
FROM payer_from_territory p
LEFT JOIN payer_all a
    ON p.payer_id = a.payer_id

UNION ALL

SELECT
    'NATIONAL_CHECK' AS check_type,
    'ALL',
    (SELECT SUM(total_elaprase_patients)
     FROM cmpa_insights_internal_schema.payer360_master
     WHERE rollup_level='TERRITORY_ALL_PAYER'),
    national_total,
    (
      (SELECT SUM(total_elaprase_patients)
       FROM cmpa_insights_internal_schema.payer360_master
       WHERE rollup_level='TERRITORY_ALL_PAYER')
      - national_total
    )
FROM national;

In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.payer360_master limit 1000;

In [0]:
select * 
from cmpa_insights_internal_schema.payer360_dq_results;